In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# Functions

In [ ]:
import os
import hashlib
import shutil

def clean_and_save_dataset(dataset_dir, output_dir):
    def hash_image(image_path):
        with open(image_path, 'rb') as f:
            return hashlib.sha256(f.read()).hexdigest()

    image_hashes = set()
    removed_files = []

    for split in ['train', 'test', 'valid']:
        image_dir = os.path.join(dataset_dir, split, 'images')
        label_dir = os.path.join(dataset_dir, split, 'labels')

        if not os.path.exists(image_dir) or not os.path.exists(label_dir):
            print(f"Skipping missing directory: {split}")
            continue

        for filename in os.listdir(image_dir):
            if not filename.lower().endswith(('.jpg', '.jpeg', '.png')):
                continue

            image_path = os.path.join(image_dir, filename)

            try:
                image_hash = hash_image(image_path)
                if image_hash in image_hashes:
                    # Remove duplicate image
                    os.remove(image_path)
                    removed_files.append(image_path)

                    # Remove corresponding label (if exists)
                    label_name = os.path.splitext(filename)[0] + '.txt'
                    label_path = os.path.join(label_dir, label_name)
                    if os.path.exists(label_path):
                        os.remove(label_path)
                else:
                    image_hashes.add(image_hash)
            except Exception as e:
                print(f"Error processing {image_path}: {e}")

    print(f"Removed {len(removed_files)} duplicate images.")

    # Save cleaned dataset
    if os.path.exists(output_dir):
        shutil.rmtree(output_dir)
    shutil.copytree(dataset_dir, output_dir)
    print(f"Cleaned dataset saved to: {output_dir}")

In [ ]:
import os
import yaml
import random
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image

def report_yolo_dataset(dataset_dir, show_examples=True, num_examples=8):
    def count_images(split):
        img_dir = os.path.join(dataset_dir, split, "images")
        if os.path.exists(img_dir):
            return len([f for f in os.listdir(img_dir) if f.lower().endswith(('.jpg', '.jpeg', '.png'))])
        else:
            return 0

    def analyze_split_labels(split, num_classes):
        label_dir = os.path.join(dataset_dir, split, "labels")
        class_image_counts = [0] * num_classes
        overlap_count = 0
        empty_or_missing_count = 0

        image_dir = os.path.join(dataset_dir, split, "images")
        if not os.path.exists(image_dir):
            return class_image_counts, overlap_count, empty_or_missing_count

        image_files = [f for f in os.listdir(image_dir) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]

        for img_file in image_files:
            img_stem = os.path.splitext(img_file)[0]
            label_path = os.path.join(label_dir, img_stem + ".txt")
            classes_in_image = set()

            if os.path.exists(label_path):
                with open(label_path, 'r') as f:
                    lines = f.readlines()
                    for line in lines:
                        parts = line.strip().split()
                        if len(parts) >= 1:
                            cls_id = int(float(parts[0]))
                            if 0 <= cls_id < num_classes:
                                classes_in_image.add(cls_id)
            else:
                empty_or_missing_count += 1
                continue  # no label file

            if len(classes_in_image) == 0:
                empty_or_missing_count += 1
            else:
                if len(classes_in_image) > 1:
                    overlap_count += 1
                for cls_id in classes_in_image:
                    class_image_counts[cls_id] += 1

        return class_image_counts, overlap_count, empty_or_missing_count

    # Print image counts
    train_count = count_images("train")
    valid_count = count_images("valid")
    test_count = count_images("test")

    print(f"📊 Image counts:")
    print(f"  Train: {train_count}")
    print(f"  Validation: {valid_count}")
    print(f"  Test: {test_count}")

    # Load class names
    data_yaml_path = os.path.join(dataset_dir, "data.yaml")
    with open(data_yaml_path, 'r') as f:
        data_yaml = yaml.safe_load(f)

    class_names = data_yaml.get("names", [])
    num_classes = len(class_names)

    print("\n🏷️ Classes in dataset:")
    for i, cls in enumerate(class_names):
        print(f"  {i}: {cls}")

    # Count per-class image occurrences and extra stats
    print("\n📂 Per-class image counts (number of images where each class appears):")
    for split in ["train", "valid", "test"]:
        class_image_counts, overlap_count, empty_count = analyze_split_labels(split, num_classes)
        print(f"  {split.capitalize()}:")
        for i, count in enumerate(class_image_counts):
            print(f"    {class_names[i]} ({i}): {count}")
        print(f"    ➕ Overlap images (contain ≥2 classes): {overlap_count}")
        print(f"    🚫 Images without any class (empty or missing labels): {empty_count}")

    if not show_examples:
        return

    # Pick examples from train split
    image_dir = os.path.join(dataset_dir, "train", "images")
    label_dir = os.path.join(dataset_dir, "train", "labels")

    image_files = [f for f in os.listdir(image_dir) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
    if len(image_files) == 0:
        print("\n⚠️ No images found in train/images to display.")
        return

    sample_files = random.sample(image_files, min(num_examples, len(image_files)))

    print(f"\n🖼️ Showing {len(sample_files)} random annotated images...")

    fig, axes = plt.subplots(2, 4, figsize=(20, 10))
    axes = axes.flatten()

    for ax, fname in zip(axes, sample_files):
        img_path = os.path.join(image_dir, fname)
        label_path = os.path.join(label_dir, os.path.splitext(fname)[0] + ".txt")

        image = Image.open(img_path)
        w, h = image.size
        ax.imshow(image)
        ax.axis('off')

        if os.path.exists(label_path):
            with open(label_path, "r") as f:
                for line in f:
                    parts = line.strip().split()
                    if len(parts) != 5:
                        continue
                    cls_id, xc, yc, bw, bh = map(float, parts)
                    x = (xc - bw / 2) * w
                    y = (yc - bh / 2) * h
                    box_w = bw * w
                    box_h = bh * h
                    rect = patches.Rectangle((x, y), box_w, box_h,
                                             linewidth=2, edgecolor='red', facecolor='none')
                    ax.add_patch(rect)
                    if class_names:
                        ax.text(x, y, class_names[int(cls_id)], color='yellow', fontsize=10,
                                bbox=dict(facecolor='black', alpha=0.5))

    plt.tight_layout()
    plt.show()


In [ ]:
import os
import shutil
import random
from tqdm import tqdm
import yaml

def filter_label_file(label_path, allowed_classes):
    """
    Filters a YOLO label file to include only allowed class IDs.
    Returns True if the file has at least one valid annotation after filtering, else False.
    """
    try:
        with open(label_path, 'r') as f:
            lines = f.readlines()

        filtered_lines = [line for line in lines if line.strip() and line.split()[0] in allowed_classes]

        if not filtered_lines:
            return False  # No valid annotations remain

        with open(label_path, 'w') as f:
            f.writelines(filtered_lines)

        return True
    except Exception as e:
        print(f"Error processing label file {label_path}: {e}")
        return False


def create_custom_dataset_by_class(
    source_dir,
    output_dir,
    class_to_keep,
    train_count,
    valid_count,
    test_count
):
    assert len(class_to_keep) == len(train_count) == len(valid_count) == len(test_count), \
        "class_to_keep, train_count, valid_count, and test_count must be the same length"

    for split in ['train', 'valid', 'test']:
        for subfolder in ['images', 'labels']:
            os.makedirs(os.path.join(output_dir, split, subfolder), exist_ok=True)

    source_yaml = os.path.join(source_dir, "data.yaml")
    output_yaml = os.path.join(output_dir, "data.yaml")

    if os.path.exists(source_yaml):
        with open(source_yaml, 'r') as f:
            data_yaml = yaml.safe_load(f)

        original_names = data_yaml.get("names", [])
        kept_class_ids = list(map(int, class_to_keep))
        kept_names = [original_names[i] for i in kept_class_ids]

        new_yaml = {
            'train': os.path.join(output_dir, 'train', 'images'),
            'val': os.path.join(output_dir, 'valid', 'images'),
            'test': os.path.join(output_dir, 'test', 'images'),
            'nc': len(kept_names),
            'names': kept_names
        }

        with open(output_yaml, 'w') as f:
            yaml.dump(new_yaml, f)

        print("✅ data.yaml generated with updated class names.")
    else:
        print("⚠️ Original data.yaml not found — skipping YAML generation.")

    # Build per-class file lists per split
    all_class_files = {cls: {'train': [], 'valid': [], 'test': []} for cls in class_to_keep}

    for split in ['train', 'valid', 'test']:
        labels_path = os.path.join(source_dir, split, 'labels')
        images_path = os.path.join(source_dir, split, 'images')
        if not os.path.exists(labels_path) or not os.path.exists(images_path):
            continue

        for label_file in os.listdir(labels_path):
            label_path = os.path.join(labels_path, label_file)

            # 🚨 Filter unwanted class annotations before anything else
            if not filter_label_file(label_path, class_to_keep):
                continue

            try:
                with open(label_path, 'r') as f:
                    lines = f.readlines()
            except Exception as e:
                print(f"Error reading {label_path}: {e}")
                continue

            classes_in_file = set(line.split()[0] for line in lines if line.strip())
            if len(classes_in_file) != 1:
                continue

            single_class = next(iter(classes_in_file))
            if single_class not in class_to_keep:
                continue

            image_filename = os.path.splitext(label_file)[0] + '.jpg'
            image_path = os.path.join(images_path, image_filename)

            all_class_files[single_class][split].append({
                'label': label_path,
                'image': image_path,
                'id': os.path.splitext(label_file)[0]
            })

    used_ids = set()

    for i, cls in enumerate(class_to_keep):
        targets = {
            'train': train_count[i],
            'valid': valid_count[i],
            'test': test_count[i]
        }

        for split in ['train', 'valid', 'test']:
            num_needed = targets[split]
            selected = []

            primary = all_class_files[cls][split]
            fallback = all_class_files[cls]['valid'] if split in ['train', 'test'] else []

            available_primary = [f for f in primary if f['id'] not in used_ids]
            take_primary = min(num_needed, len(available_primary))
            if take_primary > 0:
                chosen = random.sample(available_primary, take_primary)
                selected.extend(chosen)
                used_ids.update(f['id'] for f in chosen)
                num_needed -= take_primary

            if num_needed > 0 and split in ['train', 'test']:
                available_fallback = [f for f in fallback if f['id'] not in used_ids]
                take_fallback = min(num_needed, len(available_fallback))
                if take_fallback > 0:
                    chosen_fallback = random.sample(available_fallback, take_fallback)
                    selected.extend(chosen_fallback)
                    used_ids.update(f['id'] for f in chosen_fallback)
                    num_needed -= take_fallback

            if selected:
                print(f"📁 Copying {len(selected)} samples for class '{cls}' into '{split}'...")
                for file_pair in tqdm(selected, desc=f"[{split.upper()}] Class {cls}", unit="file"):
                    dest_label_dir = os.path.join(output_dir, split, 'labels')
                    dest_image_dir = os.path.join(output_dir, split, 'images')

                    shutil.copy(file_pair['label'], os.path.join(dest_label_dir, os.path.basename(file_pair['label'])))
                    if os.path.exists(file_pair['image']):
                        shutil.copy(file_pair['image'], os.path.join(dest_image_dir, os.path.basename(file_pair['image'])))
                    else:
                        print(f"⚠️ Missing image: {file_pair['image']}")

            if num_needed > 0:
                print(f"⚠️ Could not find {num_needed} samples for class '{cls}' in split '{split}'.")

    print(f"\n✅ Custom dataset created at: {output_dir}")


# Crosswalk

In [ ]:
!pip install roboflow

from roboflow import Roboflow
rf = Roboflow(api_key="mRQRkRQBDCgJRCWwsUYU")
project = rf.workspace("saad-pif").project("crosswalk-lqgin-rhafv")
version = project.version(1)
dataset = version.download("yolov8")


In [ ]:

clean_and_save_dataset('/content/crosswalk-1',
                       '/content/drive/MyDrive/SmartCane/clean_crosswalk5')


Removed 3997 duplicate images.
Cleaned dataset saved to: /content/drive/MyDrive/SmartCane/clean_crosswalk5


In [ ]:
# load and test datset

report_yolo_dataset("/content/crosswalk-1")


In [ ]:
# load and test datset

report_yolo_dataset("/content/drive/MyDrive/SmartCane/clean_crosswalk5")

In [ ]:
create_custom_dataset_by_class(
    source_dir='/content/drive/MyDrive/SmartCane/clean_crosswalk5',
    output_dir='/content/drive/MyDrive/SmartCane/final_crosswalk6',
    class_to_keep=['0'],
    train_count=[1000],
    valid_count=[200],
    test_count=[200]
)

✅ data.yaml generated with updated class names.
📁 Copying 1000 samples for class '0' into 'train'...


[TRAIN] Class 0: 100%|██████████| 1000/1000 [00:30<00:00, 32.48file/s]


📁 Copying 200 samples for class '0' into 'valid'...


[VALID] Class 0: 100%|██████████| 200/200 [00:06<00:00, 32.26file/s]


📁 Copying 200 samples for class '0' into 'test'...


[TEST] Class 0: 100%|██████████| 200/200 [00:06<00:00, 30.93file/s]


✅ Custom dataset created at: /content/drive/MyDrive/SmartCane/final_crosswalk6


In [ ]:
# load and test datset

report_yolo_dataset("/content/drive/MyDrive/SmartCane/final_crosswalk6")


# Pedestrain traffic light red and green

In [ ]:
!pip install roboflow

from roboflow import Roboflow
rf = Roboflow(api_key="mRQRkRQBDCgJRCWwsUYU")
project = rf.workspace("saad-pif").project("pedestrian-traffic-lights-final-orjk4")
version = project.version(4)
dataset = version.download("yolov8")



In [ ]:

clean_and_save_dataset('/content/Pedestrian-Traffic-Lights-Final-4',
                       '/content/drive/MyDrive/SmartCane/clean_rglight10')


Removed 5605 duplicate images.
Cleaned dataset saved to: /content/drive/MyDrive/SmartCane/clean_rglight10


In [ ]:
# load and test datset

report_yolo_dataset("/content/Pedestrian-Traffic-Lights-Final-4")


In [ ]:
create_custom_dataset_by_class(
    source_dir='/content/drive/MyDrive/SmartCane/clean_rglight',
    output_dir='/content/drive/MyDrive/SmartCane/final_rglight8',
    class_to_keep=['0', '1'],
    train_count=[1000, 1000],
    valid_count=[200, 200],
    test_count=[200, 200]
)

✅ data.yaml generated with updated class names.
📁 Copying 1000 samples for class '0' into 'train'...


[TRAIN] Class 0: 100%|██████████| 1000/1000 [01:46<00:00,  9.40file/s]


📁 Copying 200 samples for class '0' into 'valid'...


[VALID] Class 0: 100%|██████████| 200/200 [00:44<00:00,  4.51file/s]


📁 Copying 200 samples for class '0' into 'test'...


[TEST] Class 0: 100%|██████████| 200/200 [00:14<00:00, 13.58file/s]


... (3 outputs left out)

[VALID] Class 1: 100%|██████████| 200/200 [00:27<00:00,  7.34file/s]


📁 Copying 200 samples for class '1' into 'test'...


[TEST] Class 1: 100%|██████████| 200/200 [00:05<00:00, 35.42file/s]


✅ Custom dataset created at: /content/drive/MyDrive/SmartCane/final_rglight8


In [ ]:
# load and test datset

report_yolo_dataset("/content/drive/MyDrive/SmartCane/final_rglight8")


# Bus station

In [ ]:
!pip install roboflow

from roboflow import Roboflow
rf = Roboflow(api_key="mRQRkRQBDCgJRCWwsUYU")
project = rf.workspace("brycetjones").project("bus-stop-v3")
version = project.version(1)
dataset = version.download("yolov8")


In [ ]:
!pip install roboflow

from roboflow import Roboflow
rf = Roboflow(api_key="mRQRkRQBDCgJRCWwsUYU")
project = rf.workspace("saad-pif").project("bus-shelter-yfz47")
version = project.version(2)
dataset = version.download("yolov8")


In [ ]:
# load and test datset

report_yolo_dataset("/content/bus-stop-v3-1")


In [ ]:
# load and test datset

report_yolo_dataset("/content/Bus-Shelter-2")

In [ ]:
create_custom_dataset_by_class(
    source_dir='/content/bus-stop-v3-1',
    output_dir='/content/drive/MyDrive/SmartCane/final_bus2',
    class_to_keep=['1'],
    train_count=[400],
    valid_count=[80],
    test_count=[80]
)

✅ data.yaml generated with updated class names.
📁 Copying 400 samples for class '1' into 'train'...


[TRAIN] Class 1: 100%|██████████| 400/400 [00:09<00:00, 41.86file/s]


📁 Copying 80 samples for class '1' into 'valid'...


[VALID] Class 1: 100%|██████████| 80/80 [00:01<00:00, 41.91file/s]


📁 Copying 80 samples for class '1' into 'test'...


[TEST] Class 1: 100%|██████████| 80/80 [00:01<00:00, 40.70file/s]


✅ Custom dataset created at: /content/drive/MyDrive/SmartCane/final_bus2


In [ ]:
import os
import random
import shutil
import yaml

# Set seed for reproducibility
random.seed(42)

# Dataset sources
datasets = [
    {
        "path": "/content/drive/MyDrive/SmartCane/final_bus2",
        "class_id": 1,
        "counts": {"train": 400, "valid": 80, "test": 80}
    },
    {
        "path": "/content/Bus-Shelter-2",
        "class_id": 0,
        "counts": {"train": 600, "valid": 120, "test": 120}
    }
]

# Output dataset path
output_dir = "/content/drive/MyDrive/SmartCane/final_bus20"

# File extensions to check
extensions = [".jpg", ".png", ".jpeg"]

# Create directory if not exist
def ensure_dir(path):
    os.makedirs(path, exist_ok=True)

# Collect label files containing the target class
def collect_valid_files(label_dir, class_id):
    files = []
    for fname in os.listdir(label_dir):
        if not fname.endswith(".txt"):
            continue
        with open(os.path.join(label_dir, fname), "r") as f:
            lines = f.readlines()
        if any(line.startswith(f"{class_id} ") for line in lines):
            files.append(fname)
    return files

# Process each subset (train/valid/test)
def process_subset(source_path, class_id, count, split_name, output_root):
    src_lbl_dir = os.path.join(source_path, split_name, "labels")
    src_img_dir = os.path.join(source_path, split_name, "images")

    out_lbl_dir = os.path.join(output_root, split_name, "labels")
    out_img_dir = os.path.join(output_root, split_name, "images")
    ensure_dir(out_lbl_dir)
    ensure_dir(out_img_dir)

    valid_files = collect_valid_files(src_lbl_dir, class_id)
    if len(valid_files) < count:
        raise ValueError(f"Not enough samples for {source_path} - {split_name}")

    selected = random.sample(valid_files, count)

    for fname in selected:
        # Filter annotation lines
        with open(os.path.join(src_lbl_dir, fname), "r") as f:
            lines = f.readlines()
        new_lines = []
        for l in lines:
            parts = l.strip().split()
            if parts[0] == str(class_id):
                parts[0] = "0"  # Rename class to "shelter"
                new_lines.append(" ".join(parts))
        if new_lines:
            with open(os.path.join(out_lbl_dir, fname), "w") as f:
                f.write("\n".join(new_lines) + "\n")

            # Copy corresponding image
            base_name = fname.replace(".txt", "")
            found = False
            for ext in extensions:
                src_img_path = os.path.join(src_img_dir, base_name + ext)
                if os.path.exists(src_img_path):
                    shutil.copy(src_img_path, os.path.join(out_img_dir, base_name + ext))
                    found = True
                    break
            if not found:
                print(f"⚠️ Image not found for: {base_name}")

# Create output structure and process data
for dataset in datasets:
    for split in ["train", "valid", "test"]:
        process_subset(
            source_path=dataset["path"],
            class_id=dataset["class_id"],
            count=dataset["counts"][split],
            split_name=split,
            output_root=output_dir
        )

# Save classes.txt
with open(os.path.join(output_dir, "classes.txt"), "w") as f:
    f.write("shelter\n")

# Save dataset.yaml (YOLO format)
yaml_path = os.path.join(output_dir, "data.yaml")
yaml_dict = {
    "train": os.path.join(output_dir, "train/images"),
    "val": os.path.join(output_dir, "valid/images"),
    "test": os.path.join(output_dir, "test/images"),
    "nc": 1,
    "names": ["shelter"]
}
with open(yaml_path, "w") as f:
    yaml.dump(yaml_dict, f)

print("✅ Shelter dataset created at:", output_dir)


✅ Shelter dataset created at: /content/drive/MyDrive/SmartCane/final_bus20


In [ ]:
# load and test datset

report_yolo_dataset("/content/drive/MyDrive/SmartCane/final_bus20")


# Stairs , Escalator

In [ ]:
!pip install roboflow

from roboflow import Roboflow
rf = Roboflow(api_key="mRQRkRQBDCgJRCWwsUYU")
project = rf.workspace("saad-pif").project("escalator-stairs-h5fvy")
version = project.version(1)
dataset = version.download("yolov8")


In [ ]:

clean_and_save_dataset('/content/escalator-stairs-1',
                       '/content/drive/MyDrive/SmartCane/clean_es2')


Removed 0 duplicate images.
Cleaned dataset saved to: /content/drive/MyDrive/SmartCane/clean_es2


In [ ]:
# load and test datset

report_yolo_dataset("/content/drive/MyDrive/SmartCane/clean_es2")


In [ ]:
create_custom_dataset_by_class(
    source_dir='/content/drive/MyDrive/SmartCane/clean_es2',
    output_dir='/content/drive/MyDrive/SmartCane/final_escalator40',
    class_to_keep=['0', '1','2'],
    train_count=[1000, 1000, 1000],
    valid_count=[200, 49, 101],
    test_count=[200, 200, 200]
)

✅ data.yaml generated with updated class names.
📁 Copying 1000 samples for class '0' into 'train'...


[TRAIN] Class 0: 100%|██████████| 1000/1000 [00:28<00:00, 34.66file/s]


📁 Copying 200 samples for class '0' into 'valid'...


[VALID] Class 0: 100%|██████████| 200/200 [00:05<00:00, 34.93file/s]


📁 Copying 200 samples for class '0' into 'test'...


[TEST] Class 0: 100%|██████████| 200/200 [00:05<00:00, 34.09file/s]


... (9 outputs left out)

[VALID] Class 2: 100%|██████████| 101/101 [00:03<00:00, 28.16file/s]


📁 Copying 200 samples for class '2' into 'test'...


[TEST] Class 2: 100%|██████████| 200/200 [00:06<00:00, 30.04file/s]


✅ Custom dataset created at: /content/drive/MyDrive/SmartCane/final_escalator40


In [ ]:
# load and test datset

report_yolo_dataset("/content/drive/MyDrive/SmartCane/final_escalator40")


# Yellow line

In [ ]:
!pip install roboflow

from roboflow import Roboflow
rf = Roboflow(api_key="mRQRkRQBDCgJRCWwsUYU")
project = rf.workspace("saad-pif").project("metro-project-upiib")
version = project.version(1)
dataset = version.download("yolov8")


In [ ]:
!pip install roboflow

from roboflow import Roboflow
rf = Roboflow(api_key="mRQRkRQBDCgJRCWwsUYU")
project = rf.workspace("saad-pif").project("platform-edge-58c97")
version = project.version(1)
dataset = version.download("yolov8")


In [ ]:

clean_and_save_dataset('/content/metro-project-1',
                       '/content/drive/MyDrive/SmartCane/clean_safety1')


Removed 0 duplicate images.
Cleaned dataset saved to: /content/drive/MyDrive/SmartCane/clean_safety1


In [ ]:
# load and test datset

report_yolo_dataset("/content/metro-project-1")


In [ ]:

clean_and_save_dataset('/content/Platform-Edge-1',
                       '/content/drive/MyDrive/SmartCane/clean_safety2')


Removed 0 duplicate images.
Cleaned dataset saved to: /content/drive/MyDrive/SmartCane/clean_safety2


In [ ]:
# load and test datset

report_yolo_dataset("/content/Platform-Edge-1")


In [ ]:
import os
import random
import shutil
import yaml

# Set seed for reproducibility
random.seed(42)

# Dataset sources
datasets = [
    {
        "path": "/content/metro-project-1",
        "class_id": 1,
        "counts": {"train": 177, "valid": 24, "test": 19}
    },
    {
        "path": "/content/Platform-Edge-1",
        "class_id": 0,
        "counts": {"train": 156, "valid": 20, "test": 20}
    }
]

# Output dataset path
output_dir = "/content/drive/MyDrive/SmartCane/final_safety50"

# File extensions to check
extensions = [".jpg", ".png", ".jpeg"]

# Create directory if not exist
def ensure_dir(path):
    os.makedirs(path, exist_ok=True)

# Collect label files containing the target class
def collect_valid_files(label_dir, class_id):
    files = []
    for fname in os.listdir(label_dir):
        if not fname.endswith(".txt"):
            continue
        with open(os.path.join(label_dir, fname), "r") as f:
            lines = f.readlines()
        if any(line.startswith(f"{class_id} ") for line in lines):
            files.append(fname)
    return files

# Process each subset (train/valid/test)
def process_subset(source_path, class_id, count, split_name, output_root):
    src_lbl_dir = os.path.join(source_path, split_name, "labels")
    src_img_dir = os.path.join(source_path, split_name, "images")

    out_lbl_dir = os.path.join(output_root, split_name, "labels")
    out_img_dir = os.path.join(output_root, split_name, "images")
    ensure_dir(out_lbl_dir)
    ensure_dir(out_img_dir)

    valid_files = collect_valid_files(src_lbl_dir, class_id)
    if len(valid_files) < count:
        raise ValueError(f"Not enough samples for {source_path} - {split_name}")

    selected = random.sample(valid_files, count)

    for fname in selected:
        # Filter annotation lines
        with open(os.path.join(src_lbl_dir, fname), "r") as f:
            lines = f.readlines()
        new_lines = []
        for l in lines:
            parts = l.strip().split()
            if parts[0] == str(class_id):
                parts[0] = "0"  # Rename class to "shelter"
                new_lines.append(" ".join(parts))
        if new_lines:
            with open(os.path.join(out_lbl_dir, fname), "w") as f:
                f.write("\n".join(new_lines) + "\n")

            # Copy corresponding image
            base_name = fname.replace(".txt", "")
            found = False
            for ext in extensions:
                src_img_path = os.path.join(src_img_dir, base_name + ext)
                if os.path.exists(src_img_path):
                    shutil.copy(src_img_path, os.path.join(out_img_dir, base_name + ext))
                    found = True
                    break
            if not found:
                print(f"⚠️ Image not found for: {base_name}")

# Create output structure and process data
for dataset in datasets:
    for split in ["train", "valid", "test"]:
        process_subset(
            source_path=dataset["path"],
            class_id=dataset["class_id"],
            count=dataset["counts"][split],
            split_name=split,
            output_root=output_dir
        )

# Save classes.txt
with open(os.path.join(output_dir, "classes.txt"), "w") as f:
    f.write("safety-line\n")

# Save dataset.yaml (YOLO format)
yaml_path = os.path.join(output_dir, "data.yaml")
yaml_dict = {
    "train": os.path.join(output_dir, "train/images"),
    "val": os.path.join(output_dir, "valid/images"),
    "test": os.path.join(output_dir, "test/images"),
    "nc": 1,
    "names": ["safety-line"]
}
with open(yaml_path, "w") as f:
    yaml.dump(yaml_dict, f)

print("✅ safety-line dataset created at:", output_dir)


✅ safety-line dataset created at: /content/drive/MyDrive/SmartCane/final_safety50


In [ ]:

report_yolo_dataset("/content/drive/MyDrive/SmartCane/final_safety50")

In [ ]:
!pip install roboflow

from roboflow import Roboflow
rf = Roboflow(api_key="mRQRkRQBDCgJRCWwsUYU")
project = rf.workspace("saad-pif").project("subway-objects-mbr5v")
version = project.version(1)
dataset = version.download("yolov8")


In [ ]:

clean_and_save_dataset('/content/Subway-Objects-1',
                       '/content/drive/MyDrive/SmartCane/clean_safety8')

Removed 0 duplicate images.
Cleaned dataset saved to: /content/drive/MyDrive/SmartCane/clean_safety8


In [ ]:
report_yolo_dataset("/content/Subway-Objects-1")

In [ ]:
import os
import shutil
import yaml

def convert_segmentation_to_bbox(points):
    """
    Convert segmentation polygon points (as flat list of floats) to bounding box [x_center, y_center, width, height].
    All coordinates should already be normalized (YOLO format).
    """
    xs = points[0::2]
    ys = points[1::2]
    x_min = min(xs)
    x_max = max(xs)
    y_min = min(ys)
    y_max = max(ys)

    x_center = (x_min + x_max) / 2
    y_center = (y_min + y_max) / 2
    width = x_max - x_min
    height = y_max - y_min
    return [x_center, y_center, width, height]

def process_segmentation_dataset(dataset_path, save_path, class_id_to_keep='0', class_name='target'):
    """
    Processes a YOLOv8 instance segmentation dataset, filters to keep only the given class,
    converts polygons to bounding boxes, removes empty samples, and saves new dataset + data.yaml.
    """

    splits = ['train', 'valid', 'test']
    converted = 0
    removed_empty = 0

    for split in splits:
        labels_dir = os.path.join(dataset_path, split, 'labels')
        images_dir = os.path.join(dataset_path, split, 'images')

        if not os.path.exists(labels_dir):
            continue  # skip if split not present

        new_labels_dir = os.path.join(save_path, split, 'labels')
        new_images_dir = os.path.join(save_path, split, 'images')
        os.makedirs(new_labels_dir, exist_ok=True)
        os.makedirs(new_images_dir, exist_ok=True)

        for label_file in os.listdir(labels_dir):
            label_path = os.path.join(labels_dir, label_file)
            image_path = os.path.join(images_dir, os.path.splitext(label_file)[0] + '.jpg')

            with open(label_path, 'r') as f:
                lines = f.readlines()

            new_lines = []
            for line in lines:
                parts = line.strip().split()
                if parts[0] != class_id_to_keep:
                    continue  # Remove unwanted class

                if len(parts) < 6:
                    continue  # Invalid segmentation annotation

                try:
                    points = list(map(float, parts[1:]))  # segmentation coords
                    bbox = convert_segmentation_to_bbox(points)
                    new_line = f"{class_id_to_keep} {' '.join(f'{x:.6f}' for x in bbox)}\n"
                    new_lines.append(new_line)
                    converted += 1
                except:
                    continue  # Skip broken lines

            if new_lines:
                # Save new label
                new_label_path = os.path.join(new_labels_dir, label_file)
                with open(new_label_path, 'w') as f:
                    f.writelines(new_lines)

                # Copy image
                if os.path.exists(image_path):
                    new_image_path = os.path.join(new_images_dir, os.path.basename(image_path))
                    shutil.copy(image_path, new_image_path)
            else:
                removed_empty += 1

    # Generate data.yaml
    data_yaml = {
        'train': os.path.join(save_path, 'train', 'images'),
        'val': os.path.join(save_path, 'valid', 'images'),
        'test': os.path.join(save_path, 'test', 'images'),
        'nc': 1,
        'names': [class_name]
    }

    yaml_path = os.path.join(save_path, 'data.yaml')
    with open(yaml_path, 'w') as f:
        yaml.dump(data_yaml, f, default_flow_style=False, sort_keys=False)


    print(f"\n✅ Finished processing dataset at: {save_path}")
    print(f"📦 Converted {converted} segmentation instances to bounding boxes")
    print(f"🧹 Removed {removed_empty} images with no remaining annotations")
    print(f"📄 Created data.yaml with 1 class: {class_name}")

# Set your paths
original_path = "/content/Subway-Objects-1"         # The dataset from Roboflow
save_path = "/content/Subway-Objects-Filtered2"      # Output folder
class_id_to_keep = '0'
class_name = 'safety-line'                        # Set a friendly class name

process_segmentation_dataset(original_path, save_path, class_id_to_keep, class_name)



✅ Finished processing dataset at: /content/Subway-Objects-Filtered2
📦 Converted 525 segmentation instances to bounding boxes
🧹 Removed 4 images with no remaining annotations
📄 Created data.yaml with 1 class: safety-line


In [ ]:

report_yolo_dataset('/content/Subway-Objects-Filtered2')

In [ ]:
import os
import random
import shutil
import yaml

# Set seed for reproducibility
random.seed(42)

# Dataset sources
datasets = [
    {
        "path": "/content/Subway-Objects-Filtered2",
        "class_id": 0,
        "counts": {"train": 343, "valid": 42, "test": 43}
    },
    {
        "path": "/content/drive/MyDrive/SmartCane/final_safety50",
        "class_id": 0,
        "counts": {"train": 333, "valid": 44, "test": 39}
    }
]

# Output dataset path
output_dir = "/content/drive/MyDrive/SmartCane/final_safety5000"

# File extensions to check
extensions = [".jpg", ".png", ".jpeg"]

# Create directory if not exist
def ensure_dir(path):
    os.makedirs(path, exist_ok=True)

# Collect label files containing the target class
def collect_valid_files(label_dir, class_id):
    files = []
    for fname in os.listdir(label_dir):
        if not fname.endswith(".txt"):
            continue
        with open(os.path.join(label_dir, fname), "r") as f:
            lines = f.readlines()
        if any(line.startswith(f"{class_id} ") for line in lines):
            files.append(fname)
    return files

# Process each subset (train/valid/test)
def process_subset(source_path, class_id, count, split_name, output_root):
    src_lbl_dir = os.path.join(source_path, split_name, "labels")
    src_img_dir = os.path.join(source_path, split_name, "images")

    out_lbl_dir = os.path.join(output_root, split_name, "labels")
    out_img_dir = os.path.join(output_root, split_name, "images")
    ensure_dir(out_lbl_dir)
    ensure_dir(out_img_dir)

    valid_files = collect_valid_files(src_lbl_dir, class_id)
    if len(valid_files) < count:
        raise ValueError(f"Not enough samples for {source_path} - {split_name}")

    selected = random.sample(valid_files, count)

    for fname in selected:
        # Filter annotation lines
        with open(os.path.join(src_lbl_dir, fname), "r") as f:
            lines = f.readlines()
        new_lines = []
        for l in lines:
            parts = l.strip().split()
            if parts[0] == str(class_id):
                parts[0] = "0"  # Rename class to "shelter"
                new_lines.append(" ".join(parts))
        if new_lines:
            with open(os.path.join(out_lbl_dir, fname), "w") as f:
                f.write("\n".join(new_lines) + "\n")

            # Copy corresponding image
            base_name = fname.replace(".txt", "")
            found = False
            for ext in extensions:
                src_img_path = os.path.join(src_img_dir, base_name + ext)
                if os.path.exists(src_img_path):
                    shutil.copy(src_img_path, os.path.join(out_img_dir, base_name + ext))
                    found = True
                    break
            if not found:
                print(f"⚠️ Image not found for: {base_name}")

# Create output structure and process data
for dataset in datasets:
    for split in ["train", "valid", "test"]:
        process_subset(
            source_path=dataset["path"],
            class_id=dataset["class_id"],
            count=dataset["counts"][split],
            split_name=split,
            output_root=output_dir
        )

# Save classes.txt
with open(os.path.join(output_dir, "classes.txt"), "w") as f:
    f.write("safety-line\n")

# Save dataset.yaml (YOLO format)
yaml_path = os.path.join(output_dir, "data.yaml")
yaml_dict = {
    "train": os.path.join(output_dir, "train/images"),
    "val": os.path.join(output_dir, "valid/images"),
    "test": os.path.join(output_dir, "test/images"),
    "nc": 1,
    "names": ["safety-line"]
}
with open(yaml_path, "w") as f:
    yaml.dump(yaml_dict, f)

print("✅ safety-line dataset created at:", output_dir)


✅ safety-line dataset created at: /content/drive/MyDrive/SmartCane/final_safety5000


In [ ]:
report_yolo_dataset('/content/drive/MyDrive/SmartCane/final_safety')